# Dataset preparation, BPDR method and measured results

The research objective is to separate local pitch-repair response from uncertain note-boundary metadata while preserving the original voice and timing. This notebook follows source data, four-view preparation, model design, measured input/output and evaluation.

**Data provenance:** Vocadito is the original proposal dataset; MTG-Jamendo adds separated vocals and model-estimated metadata. The selected Transformer was pretrained on generated synthetic phrases and adapted using real Vocadito/Jamendo recordings with synthetic replay. HumTrans supplies an additional real-humming investigation.

The implementation is executable. Controlled pitch-repair and real-source noise-cleaning examples below show measured changes. These do not establish general real-singer pitch accuracy or a BPDR advantage. The complete evaluation, including unsuccessful experiments, is retained below. Running this notebook reads saved studies; it does not fit models or alter approvals.


In [ ]:
# Locate the workspace before importing project modules (JupyterLab or VS Code).
import os, sys, importlib.util
from pathlib import Path
start = Path(os.environ.get('BPDR_COMPONENT_ROOT', Path.cwd())).resolve()
COMPONENT = next((p for p in (start, *start.parents)
    if p.name == 'bpdr_input' and (p/'pyproject.toml').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Open this notebook inside BPDR_Workspace, using Run-Notebooks.ps1.')
sys.path.insert(0, str(COMPONENT/'BPDR_Workspace/source'))
required = ['numpy', 'pandas', 'matplotlib', 'soundfile', 'torch', 'IPython']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: '+', '.join(missing)+
        '. Select '+str(COMPONENT/'.venv/Scripts/python.exe')+' as the notebook kernel.')
print('Python:', sys.executable)
print('Workspace:', COMPONENT/'BPDR_Workspace')

In [ ]:
import json, hashlib, sys
from pathlib import Path
import numpy as np
import soundfile as sf
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio
from bpdr_input import COMPONENT_ROOT as COMPONENT
from bpdr_input.notebook_evidence import load_index
from bpdr_input.presentation_study import read, sha
INDEX,EVIDENCE=load_index(COMPONENT)
M=read(EVIDENCE['presentation_manifest']); RELEASE=EVIDENCE['presentation_manifest'].parent
for p,h in M['source_hashes'].items(): assert sha(p)==h, p
assert sha(EVIDENCE['presentation_manifest'])==INDEX['current_gate']['presentation_manifest_sha256']
assert sha(RELEASE/'model_card.json')==M['model_card_sha256']
CARD=read(RELEASE/'model_card.json')
assert sha(RELEASE/CARD['checkpoint'])==CARD['checkpoint_sha256']
print('Verified completed evidence; prepared hand-off:',CARD['prepared_handoff_enabled'])
display(pd.DataFrame([{'gate':k,'status':v} for k,v in M['gates'].items()]))

## 1. Proposal and dataset roles

Proposal sections 1.3 and 3.2-3.6 require a controlled comparison, not merely a working network. Vocadito is the proposed public development collection; participant research requires ethics approval and consent. Jamendo is a later exploratory source with predicted annotations.

Real-audio engineering pool: 995 training variants (77 Vocadito, 918 Jamendo), 304 development variants (19 Vocadito, 285 Jamendo). Augmentations are not independent singers. Vocadito human annotations describe performed pitch/notes, not guaranteed ideal intonation. Jamendo artist identities are proxies, not verified singer identities.

The synthetic study uses 384 training / 72 evaluation variants from 16 / 6 generated voice settings. These are not human participants. The real adaptation adds actual recordings to training; it does not certify them as approved musical references.

In [ ]:
display(pd.DataFrame([
 {'source':'Vocadito','train_variants':77,'development_variants':19,'train_groups':11,'development_groups':3,'reference':'human performed F0/notes; musical approval pending'},
 {'source':'Jamendo','train_variants':918,'development_variants':285,'train_groups':23,'development_groups':12,'reference':'separated vocals; artist proxies, model-estimated notes'},
 {'source':'Constructed humming','train_variants':384,'development_variants':72,'train_groups':16,'development_groups':6,'reference':'known construction; generated voice settings, not people'}]))
fine=COMPONENT/'runs/fine_pitch/66e3619e611241dd80d33a32578b73c6'
tr=read(fine/'training_catalog.json')['records'];dv=read(fine/'development_catalog.json')['records']
ds=lambda r:r.get('dataset','vocadito' if r['phrase_id'].startswith('vocadito') else 'jamendo')
assert not {(ds(r),r['singer_id']) for r in tr}&{(ds(r),r['singer_id']) for r in dv}
print('Train/development singer or artist-proxy groups do not overlap. Final test not accessed.')

## 2. Four views, model and losses

A = original + nominal boundaries; B = identical audio + changed boundaries. C = locally detuned audio + nominal boundaries; D = identical detuned audio + changed boundaries. Detuning is ±10/20/30 cents inside a held note with smooth ramps. Split/merge/jitter alter timing metadata only.

The model has four bidirectional Transformer layers, hidden width 128, four heads, feed-forward width 512 and dropout 0.1. Inputs contain 64 log-mel bands plus pitch, confidence, voicing, pitch change, energy, onset and a timing-boundary column (71 total). Frame-only omits that final column.

$L=L_A+\lambda_B L_B+\lambda_D L_D+\lambda_S L_S$. Here $L_A$ is squared error to zero for originals and the known compensating correction for detuned views. $L_B$ is mean absolute boundary-response difference; $L_D$ compares paired response with negative injected detuning; $L_S$ penalises abrupt correction changes. BPDR adds the response terms after 100 updates; controls use anchor + smoothness. All three use matched data, schedule and update budgets.

Fine YIN pitch replaces the earlier 33-cent-quantized PESTO pitch columns; PESTO confidence/voicing remains. Pitch is phrase-relative and normalization is training-only. This is a documented change from the original single-estimator plan. No reference target or intervention mask is an acoustic input.

In [ ]:
display(pd.DataFrame([CARD['architecture']]))
from bpdr_input.losses import LossWeights
print('Loss weights:',LossWeights())
fine_build=read(fine/'build_evidence.json')
display(pd.DataFrame(fine_build['realized_shift_training']).T[['records','median_cents','p10','p90']])

## Measured input and output

The first comparison tests local pitch correction against a **known generated reference**. It uses the selected mixed-data checkpoint and a saved audio-only run. The second uses **real HumTrans humming mixed with recorded rain**: noise is reduced using a declared room-noise prefix, then features and model inference are recalculated. Both retain original audio and provide a separate output candidate.

Pitch MAE, paired response, preservation and added-noise SNR answer different questions; there is no supported single percentage called overall model accuracy. The generated example's error is not a Vocadito/Jamendo accuracy claim. The rain mixture is a controlled example, not a naturally co-recorded novice trial.


In [ ]:
from bpdr_input.saved_results import pitch_example, noise_example
pitch_output = pitch_example(); noise_output = noise_example()
display(pd.DataFrame([
    {'task':'Local pitch correction','source':pitch_output['kind'],'input_error_cents':abs(pitch_output['before_cents']),'output_error_cents':abs(pitch_output['after_cents'])},
    {'task':'Room-noise reduction','source':noise_output['kind'],'added_noise_snr_gain_db':noise_output['snr_gain_db'],'median_pitch_change_cents':noise_output['median_pitch_change_cents']}]))
for label, sample in [('Local pitch correction',pitch_output),('Room-noise reduction',noise_output)]:
    print(label, '-', sample['kind'])
    print('Input'); display(Audio(filename=str(sample['input']), normalize=False))
    print('Processed preview'); display(Audio(filename=str(sample['output']), normalize=False))
    print(sample['scope'])
fig,axes=plt.subplots(1,2,figsize=(12,3.6))
axes[0].bar(['Input','Output'],[abs(pitch_output['before_cents']),abs(pitch_output['after_cents'])],color=['#a44324','#35735a'])
axes[0].set(ylabel='Absolute local pitch error (cents)',title='Known generated-reference example')
noise_x,noise_sr=sf.read(noise_output['input']); cleaned_x,cleaned_sr=sf.read(noise_output['output'])
assert noise_sr==cleaned_sr and noise_x.shape==cleaned_x.shape
limit=round(.7*noise_sr); ix=np.arange(0,limit,20)
axes[1].plot(ix/noise_sr,noise_x[ix],label='Input room-noise prefix',alpha=.65)
axes[1].plot(ix/noise_sr,cleaned_x[ix],label='Processed preview',alpha=.85)
axes[1].set(xlabel='Time (seconds)',ylabel='Amplitude (same scale)',title='Declared noise-only prefix'); axes[1].legend(fontsize=8)
plt.tight_layout();plt.show()


## 3. Controlled learning evaluation

Generated training phrases provide a known original and an injected local pitch change, so the required compensating correction can be measured. They complement the original datasets; they do not replace real-singer validation.

All three conditions trained for 1,500 updates per seed. Every seed is shown, including seed 340. MAE is correction error in cents. Paired response gain 1 means the model compensates for the added detuning; 0 means no detuning-specific response. Small original alteration measures preservation. Low boundary sensitivity alone is insufficient because a constant-zero model also has it. No consistent BPDR superiority is established.


In [ ]:
synth={s:read(Path(p)/'evidence.json') for s,p in M['synthetic_runs'].items()}
rows=[]
for seed,e in synth.items():
 for condition,r in e['results'].items(): rows.append(dict(seed=seed,model=condition,**r['heldout']))
synthetic_table=pd.DataFrame(rows)
display(synthetic_table[['seed','model','correction_mae','zero_mae','median_response_gain','boundary_sensitivity','unshifted_alteration']])
fig,axs=plt.subplots(1,2,figsize=(12,4))
for c in ['augmentation_only','frame_only','bpdr']:
 x=synthetic_table[synthetic_table.model==c];axs[0].plot(x.seed,x.correction_mae,'o-',label=c);axs[1].plot(x.seed,x.median_response_gain,'o-',label=c)
axs[0].axhline(19.3056,color='grey',ls=':');axs[0].set(ylabel='Correction error (cents)',xlabel='Seed')
axs[1].axhline(1,color='grey',ls=':');axs[1].set(ylabel='Paired response gain (ideal 1)',xlabel='Seed');axs[0].legend();plt.tight_layout();plt.show()

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
for c,r in synth['339']['results'].items():
 x=r['curve'];axs[0].plot([p['update'] for p in x],[p['heldout']['correction_mae'] for p in x],label=c)
 axs[1].plot([p['update'] for p in x],[p['heldout']['median_response_gain'] for p in x],label=c)
axs[0].set(xlabel='Training update',ylabel='Synthetic evaluation error (cents)');axs[1].set(xlabel='Training update',ylabel='Response gain')
axs[0].legend();plt.tight_layout();plt.show()
stats=read(EVIDENCE['presentation_statistics']);display(pd.DataFrame([dict(seed=s,control=c,**v) for s,rr in stats['synthetic'].items() for c,v in rr.items()]))
print('95% intervals resample synthetic voice settings, not people. Seeds remain separate.')

## 4. Does it rely on the longest-note shortcut?

The original constructed training always shifts the longest note. A new frozen-model challenge shifts a nonlongest note in each of 24 phrases, with eight new generated voice settings. All cases are retained; no model is refitted on this challenge. It probes one confound, not the full synthetic-to-real gap.

In [ ]:
probe=read(EVIDENCE['presentation_probe'])
display(pd.DataFrame([dict(model=c,**r['all']) for c,r in probe['results'].items()]))
fig,ax=plt.subplots(figsize=(7,3.5));names=list(probe['results']);ax.bar(names,[probe['results'][c]['all']['correction_mae'] for c in names])
ax.axhline(20,color='grey',ls=':',label='No correction');ax.set(ylabel='Correction error (cents)',title='New synthetic phrases: shifted note is never longest');ax.legend();plt.xticks(rotation=12);plt.tight_layout();plt.show()

## 5. Use of real Vocadito and Jamendo recordings

First, all nine frozen model/seed combinations were evaluated on the same 304 real-audio development variants, without fitting or refitting normalization. Engineering gate: each dataset must have median paired response ≥0.25, original alteration ≤1 cent, and positive correction gain. This is a development diagnostic, not a final test.

Second, a bounded transfer-learning attempt uses 995 real training variants plus 384 synthetic variants: 300 new updates per model, seed 339, two Vocadito + two Jamendo + four synthetic draws per update, learning rate 0.0001. The pretrained normalizer is frozen; BPDR response losses start after 100 updates. Train/development groups are separate. All three final checkpoints and failures are kept. There is no development early stopping or extra retry budget.

In [ ]:
transfer=read(EVIDENCE['presentation_transfer']);adapt=read(EVIDENCE['presentation_adapt']);rr=[]
for seed,conditions in transfer['results'].items():
 for c,result in conditions.items():
  for ds,s in result['by_dataset'].items():rr.append(dict(stage='frozen synthetic',seed=seed,model=c,dataset=ds,passes=result['engineering_gate_passes'],**s))
for c,result in adapt['results'].items():
 for ds,s in result['by_dataset'].items():rr.append(dict(stage='real + synthetic adaptation',seed='339',model=c,dataset=ds,passes=result['engineering_gate_passes'],**s))
real_table=pd.DataFrame(rr)
display(real_table[['stage','seed','model','dataset','records','correction_mae','median_response_gain','unshifted_alteration','passes']])
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,ds in zip(axs,['vocadito','jamendo']):
 names=['augmentation_only','frame_only','bpdr'];y=np.arange(3)
 for offset,stage in [(-.18,'frozen synthetic'),(.18,'real + synthetic adaptation')]:
  x=real_table[(real_table.dataset==ds)&(real_table.seed=='339')&(real_table.stage==stage)].set_index('model')
  ax.barh(y+offset,[x.loc[c,'median_response_gain'] for c in names],height=.32,label=stage)
 ax.set_yticks(y,['Augmentation','Frame-only','BPDR']);ax.axvline(.25,ls=':',color='grey');ax.set(title=ds,xlabel='Median paired response gain; gate 0.25');ax.legend(fontsize=8)
plt.tight_layout();plt.show()
display(pd.DataFrame([dict(seed=s,dataset=ds,control=c,**v) for s,dd in stats['real_transfer'].items() for ds,rr in dd.items() for c,v in rr.items()]))

In [ ]:
display(pd.DataFrame([dict(dataset=ds,control=c,**v) for ds,rr in M['adaptation_paired_intervals'].items() for c,v in rr.items()]))
display(pd.DataFrame([dict(model=c,**v) for c,v in M['adaptation_synthetic_retention'].items()]))
display(pd.DataFrame([dict(model=c,**v) for c,v in M['adaptation_nonlongest_challenge'].items()]))
print('Adaptation intervals use the same real groups; one adaptation seed only, no training-stability claim.')

### Five proposal conditions and preservation

Original audio is the zero-correction reference. The existing conservative rule estimates the singer's tuning offset and requests constant corrections only on stable held notes. The table below scores its raw requested curve on the same 304 variants as the neural models. It uses existing fine-pitch archives and fixed thresholds, without calibration. These prediction scores are not rendered-audio results or a matched-acceptance-coverage study. Musical harmful-edit rate remains unknown without independent references/listening.

In [ ]:
baselines=read(EVIDENCE['presentation_baselines']);five=[]
for c,r in baselines['results'].items():
 for ds,s in r['by_dataset'].items():five.append(dict(model=c,dataset=ds,**s))
for c,r in transfer['results']['339'].items():
 for ds,s in r['by_dataset'].items():five.append(dict(model=c,dataset=ds,**s))
display(pd.DataFrame(five)[['model','dataset','correction_mae','median_response_gain','unshifted_alteration','outside_alteration']])
old=read(COMPONENT/'runs/rule_evaluation/b3c92d055b7e45e48e553ad9a796c726/evidence.json')
display(pd.DataFrame(old['summary']).T[['originals','originals_with_any_edit','detuned','detuned_shift_edited']])
print('Earlier rule decisions are separate engineering evidence; no musical harmful-edit rate is inferred.')

## 6. End-to-end inference and preservation

The selected-checkpoint input/output examples appear after the method section. Audio-only runtime derives boundaries from the waveform and does not receive reference notes, the clean target or intervention masks. It compares nominal, jittered and removed-boundary views before permission, rendering, remeasurement and rollback checks.

Valid zero-cent A4 pitch uses explicit voicing. Historical caches/results are retained; a later rebuild is evaluated separately. The accepted downstream working copy preserves original pitch/timing, while verified processing candidates can be played and downloaded separately.


In [ ]:
display(pd.DataFrame(M['runtime_checks']))
display(read(COMPONENT/'runs/presentation_selected_example_measurement.json'))
row=M['runtime_checks'][0]; folder=Path(row['path'])
with np.load(folder/'model_predictions.npz') as z:
    fig,ax=plt.subplots(figsize=(11,3.5))
    for i,label in enumerate(['Nominal','Jittered','Removed edges']):ax.plot(z['times'],z['predictions'][i],label=label)
ax.set(xlabel='Time (seconds)',ylabel='Requested correction (cents)',title='Audio-only inference: no private reference supplied')
ax.legend();plt.tight_layout();plt.show()


## 7. Reliability, timing and reproducibility

The technical path distinguishes acceptance, re-record requests and processing failures; failed inputs have no accepted audio. The controller withholds excessive/uncertain edits and rolls back failed candidates. Rendering preserves sample count and copies samples outside approved support. These are tested engineering properties, not evidence of musical quality.

The following timing sample repeats one constructed 20-second phrase five times on the recorded Windows device, with technical analysis and model inference, preview rendering off. The p95 is a descriptive sample percentile; it does not establish concurrent-service performance or another OS.
The sealed-release timing/test values below belong to that earlier snapshot. Current completion checks include 448 regression tests, a fresh UI check and notebook re-execution.


In [ ]:
display(M['latency']['hardware']);display(pd.DataFrame(M['latency']['rows']))
display(M['training_environment']);display(pd.DataFrame(M['contract_checks']['rows']))
print('Median seconds:',M['latency']['median_seconds'],'Sample p95:',M['latency']['p95_seconds'],'30-second target:',M['latency']['passes'])
fig,ax=plt.subplots(figsize=(7,3));ax.bar(range(1,6),[r['seconds'] for r in M['latency']['rows']]);ax.axhline(30,c='grey',ls=':');ax.set(xlabel='Warm repeat after three runtime checks',ylabel='Elapsed seconds',title='Constructed 20-second phrase, inference preview');plt.tight_layout();plt.show()
print('Shared pipeline Python files verified unchanged:',M['shared_pipeline_unchanged'])
print('Regression receipt:',COMPONENT/'runs/presentation_full_regression.log')
print('Local interface:',COMPONENT/'Run-System.ps1')

## 8. Why synthetic success does not imply real-singer success

These are two different questions: was the input technically usable, and did the model respond correctly to our added local detuning? Real recordings are not all rejected as files. The selected checkpoint already includes Vocadito/Jamendo adaptation, but its real paired response is approximately zero. That is a model-generalisation failure, not evidence of bad music.

The table describes original performed-pitch features, including transitions and tracker variation. It does not score musicality. Correlated variants are not independent people. Real contours move more, and Vocadito has much shorter labelled correction coverage than the synthetic controls. Preservation targets are more numerous, which may favour a near-zero solution; this is a hypothesis, not an isolated causal result. Synthetic training also had a longest-note shortcut and a failed seed.

An additional concrete defect in historical archives treats valid zero-cent A4-bin pitch as missing. The audit below finds this in Jamendo, with **zero affected Vocadito frames**: it cannot explain both failures. Live runtime v2 already fixes voicing; the old training archives and model weights remain frozen pending a versioned rebuild and new declared pilot. Original/zero outputs are not learned recovery.

The versioned A4 rebuild and bounded learning check have now been executed; see section 11. Correcting this implementation defect did not establish real-data learning.


In [ ]:
diagnosis=read(EVIDENCE['real_input_diagnosis'])
display(pd.DataFrame([dict(source=k,variants=v['variants'],**v['medians']) for k,v in diagnosis['summary'].items()]))
display(pd.DataFrame([dict(source=k,**v) for k,v in diagnosis['legacy_adapter'].items()]))


## 9. Conditional room-noise cleaning: separate engineering gate

Proposal p. 26 permits limited noise reduction only with a reliable noise-only interval, retaining original phase, rejecting vocal damage and recalculating features. The optional preview requires a caller-declared 0.7-second room-noise prefix without humming, speech, breaths or music. Broadband/stationarity/periodicity screening may reject it; this is not a speech/noise classifier and cannot prove that the declaration is correct.

The fixed preview uses a smoothed spectral magnitude mask capped at 6 dB attenuation. It preserves sample count and uses original phase. PESTO/independent YIN checks assess pitch, voiced-frame loss and voice attenuation. These are provisional numerical safeguards, not perceptual or musical approval. No reliable reference means no noise processing; overlapping speech and changing noise are unsupported.

A protocol froze eight training-only sources (three Vocadito, three Jamendo, two generated voices), two noise proxies and two SNR levels before execution: 32 correlated controls, not 32 users. The baseline is the recording **before added noise**, not a certified clean vocal. Score improvement against that baseline, including active-source samples, and retain every failure. No denoiser or Transformer was trained; no final-test access.

**Result:** 13/24 real-audio controls and 8/8 synthetic controls passed. Passed real controls have median overall SNR improvement 4.01 dB and active-source improvement 2.94 dB. Eleven real controls were withheld despite improved SNR because the voice checks failed. This is a useful cleaner-output demonstration on real source audio; it does not demonstrate real pitch repair or robustness on naturally noisy novice recordings.

This section preserves the first white/fan-proxy policy experiment. The current app uses the separate 3-dB colored-reference policy evaluated in section 10. Earlier thresholds and results are not rewritten.


In [ ]:
noise=read(EVIDENCE['noise_study'])
noise_rows=[dict(dataset=r['source']['dataset'],noise=r['noise'],snr=r['nominal_snr_db'],passes=r['passes'],snr_gain_db=r['snr_gain_db'],active_snr_gain_db=r['active_source_snr_gain_db'],reasons=', '.join(r['audit']['reasons'])) for r in noise['rows']]
display(pd.DataFrame([dict(dataset=k,**v) for k,v in noise['summary'].items()]))
display(pd.DataFrame(noise_rows))
fig,ax=plt.subplots(figsize=(10,3))
for passed,color,label in [(True,'#35735a','Numerical preservation passed'),(False,'#b55732','Withheld')]:
 ix=[i for i,r in enumerate(noise_rows) if r['passes']==passed]
 ax.scatter(ix,[noise_rows[i]['snr_gain_db'] for i in ix],color=color,label=label)
ax.set(xlabel='Control index (correlated variants)',ylabel='SNR improvement (dB)',title='Noise reduction alone is insufficient: preservation must also pass')
ax.legend();plt.show()
example=next(r for r in noise['rows'] if r['passes'] and r['source']['dataset']=='vocadito')
for name,label in [('baseline.wav','Real Vocadito baseline before added noise'),('noisy.wav','Same recording with known added noise'),('candidate.wav','Separate noise-cleaning candidate')]:
 print(label);display(Audio(filename=str(Path(example['directory'])/name)))
runtime=read(EVIDENCE['noise_runtime']);display({k:runtime[k] for k in ['outcome','model_outcome','prepared_handoff_enabled']})
assert runtime['noise_candidate']['features_recalculated'] and runtime['handoff']['status']=='ACCEPT_UNCHANGED'


## 10. Colored room-noise policy and fixed downstream diagnostic

The first policy's high spectral-flatness requirement rejected a steady recorded rain reference even though its periodicity/stationarity checks passed. A separately versioned policy allows a more colored reference (flatness floor .01) and **reduces** maximum attenuation from 6 to 3 dB. Pitch, voiced-frame and voice-attenuation limits are unchanged. The caller must still declare a genuine noise-only interval; no automatic speech/noise classification is claimed.

The policy was fixed after the first-fold screening diagnosis and checked on previously unused ESC-50 fold-2 noises. Six training-source excerpts (two each from Vocadito, Jamendo and HumTrans) were mixed with recorded rain/vacuum/clock sounds at 10/20-dB nominal SNR: 36 controls. Six additional breathing controls had **no eligible declared noise reference** and must stay unchanged; they do not prove breath detection. These are constructed mixtures of real recordings, not naturally co-recorded novice/clean pairs.

For a matched comparison, both policies were run on the same fold-2 mixtures. The original policy passed 6/24 rain/vacuum controls; the revised policy passed 15/24. All clock controls were withheld. Keep failures, source correlation and limited noise diversity explicit. The app now uses the revised policy for its optional preview, always retaining original downstream audio.

A real HumTrans/rain preview also completed re-analysis and inference. A separate CPU check ran the pinned component-local GAME model on baseline, repeated baseline, noisy and cleaned preview. Baseline-repeat onset+pitch agreement was 1.0; noisy and cleaned agreement were both 0.9444. This is one extractor-consistency diagnostic, **not human transcription accuracy or evidence of an extraction gain**. No shared extraction pipeline was executed or changed.


In [ ]:
field=read(EVIDENCE['environmental_noise_v2']);paired=read(EVIDENCE['noise_policy_comparison'])
display(pd.DataFrame([dict(noise=k,**v) for k,v in field['summary'].items()]))
display(pd.DataFrame(paired['rows']))
fig,ax=plt.subplots(figsize=(9,3))
keys=list(field['summary']);counts=[field['summary'][k]['passes'] for k in keys];totals=[field['summary'][k]['cases'] for k in keys]
ax.bar(keys,counts,color='#35735a',label='Numerical preservation passed')
ax.bar(keys,np.array(totals)-counts,bottom=counts,color='#d5b49a',label='Withheld / identity control')
ax.set(ylabel='Correlated controls',title='Recorded-noise mixtures: preserve the voice or withhold')
ax.legend();plt.show()
live=read(EVIDENCE['humming_noise_runtime']);case=Path(live['source_study_case']['directory'])
for label,path in [('Before added noise',case/'baseline.wav'),('Real humming + recorded rain',case/'noisy.wav'),('Separate noise-cleaned preview',Path(live['report']['noise_candidate']['path']))]:
 print(label);display(Audio(filename=str(path)))
game=read(EVIDENCE['completion_handoff']);display(pd.DataFrame([dict(input=k,**v) for k,v in game['scores'].items()]))
assert live['report']['handoff']['status']=='ACCEPT_UNCHANGED'


## 11. Versioned A4-safe inputs and real-learning check

All 995 real training and 304 development variants were rebuilt in a new run. Valid zero-cent A4-bin pitch is no longer treated as missing; the 50-cent YIN/PESTO agreement guard remains. Other raw acoustic/boundary columns and original target files are preserved. Training-only hierarchical moments are fitted after explicit-voicing phrase centering; declared pitch/pitch-change scales are 100/30 cents. Restored-frame totals count views and correlated variants, not unique people.

Before broad training, eight fixed real training variants were fitted for 200 updates with equal correction/preservation exposure inside C/D, preserved A/B anchors, and output-gain30 optimization. An exact linear remap preserves the synthetic initialization's function under the new normalizer; the saved head is converted back to physical cents. This is a documented anchor/optimization change, not a new claimed method advantage.

The check failed: Jamendo response remained approximately zero, Vocadito response about .0028, and Vocadito original alteration about 1.18 cents. The declared larger matched fit therefore remained locked. We report a bounded null result rather than silently extending training or changing acceptance thresholds.


In [ ]:
rebuilt=read(EVIDENCE['real_v2_build']);pilot=read(EVIDENCE['real_v2_pilot'])
display(rebuilt['counts']);display(pd.DataFrame([dict(dataset=k,**v) for k,v in pilot['results']['augmentation_only']['by_dataset'].items()]))
assert not pilot['training_gate_passes']


## 12. Real humming intake, suitability and bounded training

[HumTrans](https://huggingface.co/datasets/dadinghh2/HumTrans) is now used in component-owned intake, audio preparation, known-error pair generation, evaluation and a separate training attempt. Source card, CC BY-NC 4.0 attribution, pinned revision, file hashes, raw WAVs and MIDI are retained. Standard HTTP byte ranges retrieved the selected audio without downloading the 14.7-GB archive.

The initial 16 recordings used distinct compositions and separate filename-person groups: six training people, two development people, two reserved unseen people. All selected audio came from **official TRAIN**. The official train/valid/test metadata includes all ten people in each split, so it does not by itself establish unseen-person performance. The component's small split is intentionally separate from that benchmark. Its sorted-person choice is not a representative demographic sample.

Initially only two training recordings yielded qualified long-note pairs, with no qualifying development pair. A second, prospectively declared sample chose eight new compositions for the same development people using score-held-note duration **before audio/model scoring**. It yielded one qualified development recording. All 24 recordings, including 21 with no usable audited pair, remain reported. The final catalog has 12 training variants from two recordings/people and six development variants from one recording/person. Variants are not independent people; this selected subset cannot support population accuracy or a BPDR advantage.

MIDI stays diagnostic-only. Lag/octave discrepancies are reported without rewriting the score, forcing the singer to a reference or feeding a guide into inference. The [authors](https://arxiv.org/html/2309.09623v2) collected guided Da-Da-Da recordings from musically experienced participants and filtered poor/noisy takes. This does not validate noisy, inexperienced, freely invented or every closed-mouth humming style. A [later transcription study](https://github.com/shubham-gupta-30/humming_transcription) also reports timing-label problems; its automatic corrections are not human BPDR approvals.

The frozen selected model showed approximately zero recovery on the six available humming development variants. A declared 200-update BPDR attempt then mixed four sign-balanced real humming variants with four synthetic training variants per update. Training normalization, fixed schedule, initialization/remapping, endpoint-only scoring and all failures are saved. No development early stopping, final-test audio or review approval was used. The candidate improved training recovery but failed development preservation and degraded synthetic correction, so **it was not selected**.


In [ ]:
intake=read(EVIDENCE['humming_intake']);prepared=read(EVIDENCE['humming_preparation']);humfit=read(EVIDENCE['humming_learning']);frozen=read(EVIDENCE['humming_frozen_probe'])
display(prepared['summary'])
display(pd.DataFrame([dict(recording=r['key'],split=r['split'],person=r['person_id'],status=r['status'],seconds=r['duration_seconds'],voiced_fraction=r['voiced_fraction']) for r in prepared['sources']]))
display(pd.DataFrame([dict(split=k,**v['all']) for k,v in humfit['results'].items()]))
display(humfit['gates']);display(frozen['results']['by_dataset'])
# Point measurements only: one selected development person does not support a population interval.
fig,ax=plt.subplots(figsize=(9,3))
labels=list(humfit['results']);ax.bar(labels,[humfit['results'][k]['all']['correction_mae'] for k in labels],color=['#9c6247','#b55732','#35735a'])
ax.plot(labels,[humfit['results'][k]['all']['zero_mae'] for k in labels],marker='o',color='#182f32',label='No-correction error')
ax.set(ylabel='Requested correction MAE (cents)',title='Humming training improved; development did not generalise')
ax.tick_params(axis='x',labelrotation=12);ax.legend();plt.show()
decision=read(EVIDENCE['completion_model_decision']);display(decision)
assert not decision['research_model_adopted']


## 13. Implemented behaviour and validation scope

**Implemented:** dataset analysis and grouped preparation; four-view Transformer and matched conditions; original retention and technical stops; audio-only inference; permission, rendering, remeasurement and rollback; optional declared-reference noise cleaning with feature re-analysis; local recording/upload; reproducible results, checkpoint selection and complete experiment records.

**Measured outcomes:** the selected model retains controlled generated-phrase pitch repair after real-data adaptation. Some real-source added-noise mixtures improve while passing numerical preservation checks. Inputs that need no supported change remain unchanged. The model is not required to alter every recording.

**Validation still open:** reliable real-voice pitch recovery, a consistent BPDR advantage, independent musical reference review, novice-user and blinded listening studies, downstream extraction improvement and a reserved final evaluation. The real-input and HumTrans candidates did not pass their checks and were not selected. Research-ready reference approvals remain zero; prepared downstream handoff stays off.

The full chronological evidence remains in `runs/`, with the current method and data provenance explained above. Future work needs a newly declared learning design and stronger independently reviewed real/humming coverage. Lower training error or a changed waveform alone does not establish useful generalisation.
